# SageMaker Fine-Tuning Lab

Part 2 of 3 · Model Fine-Tuning, Evaluation and Deployment with AWS SageMaker

Run a real LoRA fine-tuning job on Amazon SageMaker, save and inspect the model artifact, deploy an endpoint and send it a request.

**You will learn to:**

- Describe the SageMaker training and deployment workflow
- Select a supported model and compute configuration
- Upload a prepared dataset and configure a fine-tuning job
- Run, monitor and inspect a training job and its artifact
- Deploy a model to a real-time endpoint and invoke it
- Clean up resources to stop charges

> **Before you start:**
> - Finish `01_Fine_Tuning_Fundamentals.ipynb` so the `data/` folder exists.
> - Run this notebook in **SageMaker Studio** (or a SageMaker notebook instance), or on your laptop with a `.env` file that holds `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION` and `SAGEMAKER_ROLE_ARN`.
> - Your instructor must have raised the instance quotas and accepted the model licence ahead of time (see the pre-work checklist).
>
> **Cost warning:** GPU instances and endpoints bill by the hour. Run the **Clean up** section at the end, even if something fails.
>
> **SDK version:** this lab uses the SageMaker Python SDK **v2** classes (`JumpStartEstimator`). The install cell pins `sagemaker<3`.

> **How to run this notebook**
> - Run the cells from top to bottom, one at a time.
> - Cells that start a job or an endpoint are safe to run twice: they check first, so you do not get a second job or a duplicate endpoint.
> - This account allows **one training job and two endpoints at a time**. AWS can also be out of free GPUs for a while; the job then waits in `Pending` (not billed). The Troubleshooting section lists what to do.

## 1. SageMaker workflow at a glance

```
 Your notebook                     Amazon S3                       SageMaker
 -------------                     ---------                       ---------
 data/train/*.jsonl  ---upload--->  s3://bucket/.../train/
                                          |
                                          v
                                  (1) Training job  <---- model + container (JumpStart)
                                      runs on a GPU instance (ml.g5.*), logs to CloudWatch
                                          |
                                          v
                                   s3://bucket/.../output/model artifact
                                          |
                                          v
                                  (2) Model -> Endpoint config -> Endpoint (real-time)
                                          |
                                          v
 predictor.predict(...)  <----------  HTTPS invoke  <----------  (3) Monitoring (CloudWatch)
```

| Piece | Role |
|---|---|
| Execution role | The IAM identity SageMaker uses to read S3 and run jobs |
| S3 bucket | Stores datasets and model artifacts |
| Training job | A temporary GPU container that runs the fine-tuning and then stops |
| Model artifact | The saved result (for LoRA, mainly adapter weights and config) |
| Endpoint | A running instance that answers requests; **billed while it exists** |
| CloudWatch | Logs and metrics for jobs and endpoints |

## 2. Setup

In [1]:
%%capture
%pip install --upgrade "sagemaker<3" boto3 pandas python-dotenv

In [2]:
import os, logging, warnings
os.environ["SAGEMAKER_SUPPRESS_V2_WARNING"] = "1"      # silences the v2 deprecation notices
for noisy in ("sagemaker.config", "sagemaker.jumpstart", "sagemaker.telemetry", "botocore.credentials"):
    logging.getLogger(noisy).setLevel(logging.ERROR)     # hides routine INFO/WARNING lines, keeps real errors
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=SyntaxWarning)

In [ ]:
import os, json, time, pathlib
from dotenv import load_dotenv
load_dotenv(override=True)         # use the AWS keys in .env (not stale keys from ~/.aws); must come before boto3 / sagemaker

import boto3, sagemaker
from botocore.exceptions import ClientError

boto_sess = boto3.Session(region_name=os.getenv("AWS_DEFAULT_REGION"))
sess = sagemaker.Session(boto_session=boto_sess)
region = sess.boto_region_name

# The role SageMaker assumes to read S3 and run jobs.
# Laptop: set SAGEMAKER_ROLE_ARN in .env.  SageMaker Studio: it is found automatically.
role = os.getenv("SAGEMAKER_ROLE_ARN")
if not role:
    try:
        role = sagemaker.get_execution_role()
    except Exception:
        role = None
assert role, "Set SAGEMAKER_ROLE_ARN in your .env file (the ARN of the SageMaker execution role)."

bucket = sess.default_bucket()     # created if it does not exist yet
sm = boto_sess.client("sagemaker")
s3 = boto_sess.client("s3")
caller = boto_sess.client("sts").get_caller_identity()
assert caller["Account"] == role.split(":")[4], "The role belongs to a different AWS account than your login."

print("Region :", region)
print("Bucket :", bucket)
print("Role   :", role.split("/")[-1])
print("Login  :", caller["Arn"].split(":")[-1])

Two different identities are involved. **Your login** (an IAM user or role) calls the AWS APIs from this notebook. The **execution role** is the one SageMaker assumes to read S3 and run the training job. That is why `SAGEMAKER_ROLE_ARN` points at the execution role and not at your login.

In [4]:
from sagemaker.predictor import Predictor
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

def job_status(name=None):
    """(status, secondary status, failure reason) of a training job."""
    d = sm.describe_training_job(TrainingJobName=name or JOB_NAME)
    return d["TrainingJobStatus"], d["SecondaryStatus"], d.get("FailureReason", "")

def jobs_with(status):
    return [j["TrainingJobName"] for p in sm.get_paginator("list_training_jobs").paginate(StatusEquals=status)
            for j in p["TrainingJobSummaries"]]

def all_endpoints():
    return [(e["EndpointName"], e["EndpointStatus"]) for p in sm.get_paginator("list_endpoints").paginate()
            for e in p["Endpoints"]]

def endpoint_status(name):
    """(status, failure reason). Status is None when the endpoint does not exist."""
    if not name:
        return None, ""
    try:
        d = sm.describe_endpoint(EndpointName=name)
        return d["EndpointStatus"], d.get("FailureReason", "")
    except ClientError:
        return None, ""

def name_is_free(name):
    """True if no endpoint and no endpoint config uses this name (a failed deploy can leave a config behind)."""
    if endpoint_status(name)[0] is not None:
        return False
    try:
        sm.describe_endpoint_config(EndpointConfigName=name)
        return False
    except ClientError:
        return True

def get_predictor(name):
    return Predictor(endpoint_name=name, sagemaker_session=sess,
                     serializer=JSONSerializer(), deserializer=JSONDeserializer())

def wait_for_endpoint(name, timeout_min=30):
    """Wait until the endpoint is InService. Returns True on success, False on failure or timeout."""
    start = time.time()
    while True:
        status, reason = endpoint_status(name)
        print(time.strftime("%H:%M:%S"), name, "|", status)
        if status == "InService":
            return True
        if status in (None, "Failed"):
            print("Endpoint problem:", (reason or "endpoint not found")[:300])
            return False
        if time.time() - start > timeout_min * 60:
            print(f"Still not ready after {timeout_min} minutes. It keeps starting in the background.")
            return False
        time.sleep(30)

## 3. Select a supported model and compute

Set the model and instance in one place. **Confirm the model ID in your region** (SageMaker Studio -> JumpStart, or the model catalog) and confirm the instance type is supported for fine-tuning of that model. IDs and supported instances change over time.

In [5]:
MODEL_ID = "meta-textgeneration-llama-3-2-1b-instruct"   # verify in your region
MODEL_VERSION = "2.19.0"
# JumpStart fine-tuning supports: ml.g5.2xlarge (default), 
# ml.g5.4xlarge, ml.g5.8xlarge, ml.g5.12xlarge, ml.g4dn.12xlarge ...
TRAIN_INSTANCE = "ml.g5.2xlarge"                     
# Hosting a 1B model works on smaller GPUs too. 
# If AWS has no free GPU, try ml.g5.xlarge, ml.g6.2xlarge or ml.g5.4xlarge.
ENDPOINT_INSTANCE = "ml.g5.xlarge"

# One run id for this whole run (S3 folder and endpoint names).
# Running this cell again keeps the same id. To start a completely new run, set RUN_ID = None first.
RUN_ID = globals().get("RUN_ID") or time.strftime("%Y%m%d-%H%M")
PREFIX = "finetune-lab"
TUNED_ENDPOINT = globals().get("TUNED_ENDPOINT") or f"lab-tuned-{RUN_ID}"
print(MODEL_ID, MODEL_VERSION, TRAIN_INSTANCE, "| run", RUN_ID)

meta-textgeneration-llama-3-2-1b-instruct 2.19.0 ml.g5.2xlarge | run 20261126-1150


**Check the quota first.** A common failure is `ResourceLimitExceeded`. This cell lists your quotas for the instance type (it may need extra permissions; if it errors, check **Service Quotas** in the console instead).

In [6]:
try:
    sq = boto_sess.client("service-quotas")
    for page in sq.get_paginator("list_service_quotas").paginate(ServiceCode="sagemaker"):
        for q in page["Quotas"]:
            if TRAIN_INSTANCE in q["QuotaName"] and ("training job usage" in q["QuotaName"] or "endpoint usage" in q["QuotaName"]):
                print(f"{q['QuotaName']:<55} {q['Value']}")
except Exception as error:
    print("Could not read quotas here:", repr(error)[:150])

ml.g5.2xlarge for training job usage                    1.0
ml.g5.2xlarge for spot training job usage               1.0
ml.g5.2xlarge for endpoint usage                        2.0


**Then check what is already using that quota.** If you see jobs or endpoints from an earlier attempt, run the **Clean up** cell at the end of this notebook first, otherwise the new job or endpoint is refused with `ResourceLimitExceeded`.

In [7]:
print("Training jobs running :", jobs_with("InProgress") + jobs_with("Stopping") or "none")
print("Endpoints             :", all_endpoints() or "none")

Training jobs running : none
Endpoints             : none


## 4. Load the prepared dataset and upload it

In [8]:
DATA_DIR = pathlib.Path("data")
assert (DATA_DIR / "train" / "train.jsonl").exists(), "Run Notebook 1 first."
assert (DATA_DIR / "train" / "template.json").exists(), "template.json is missing."

for split in ["train", "validation", "test"]:
    f = DATA_DIR / split / f"{split}.jsonl"
    print(f"{split:<11} {sum(1 for _ in open(f)):>5} rows")

print("\nFirst training row:")
print(open(DATA_DIR / "train" / "train.jsonl").readline())

train         936 rows
validation     98 rows
test           98 rows

First training row:
{"instruction": "Classify the intent of this message. Choose one of: card_arrival, card_payment_fee_charged, cash_withdrawal_charge, declined_card_payment, exchange_rate, lost_or_stolen_card, pin_blocked, top_up_failed, transfer_not_received_by_recipient, verify_my_identity.", "context": "Can you tell me why I haven't received my new card yet?", "response": "card_arrival"}



In [9]:
train_uri = sess.upload_data(str(DATA_DIR / "train"), bucket=bucket, key_prefix=f"{PREFIX}/{RUN_ID}/train")
val_uri = sess.upload_data(str(DATA_DIR / "validation"), bucket=bucket, key_prefix=f"{PREFIX}/{RUN_ID}/validation")
print("train     :", train_uri)
print("validation:", val_uri)

# Check the files really arrived (train.jsonl, template.json, validation.jsonl)
uploaded = [o["Key"] for o in s3.list_objects_v2(Bucket=bucket, Prefix=f"{PREFIX}/{RUN_ID}/").get("Contents", [])]
assert len(uploaded) >= 3, f"Upload incomplete, found only: {uploaded}"
print("Uploaded files:", [k.split(f"{PREFIX}/{RUN_ID}/")[-1] for k in uploaded])

train     : s3://sagemaker-us-east-1-018913575808/finetune-lab/20261126-1150/train
validation: s3://sagemaker-us-east-1-018913575808/finetune-lab/20261126-1150/validation
Uploaded files: ['train/template.json', 'train/train.jsonl', 'validation/validation.jsonl']


The training job reads only the `train` folder (the channel is called `training`). This model holds out 20% of it for validation (`validation_split_ratio`), so the separate validation file is uploaded for reference.

## 5. Configure the fine-tuning job

`JumpStartEstimator` packages the model, the training container and default settings for the chosen model. Print the defaults first, then override only what you need.

> **Why `epoch = 3` and `learning_rate = 0.0002`?** This is a demonstration run on **one GPU**. In our tests a single pass gave too few update steps for the model to change how it answers (it kept replying in sentences instead of a label). Three passes give roughly 560 update steps, which keeps the job short (about 10 minutes) and cheap (about 30 cents).
>
> **Stay on a one-GPU instance** for this lab. On a multi-GPU instance every step uses more examples, so there are fewer steps and the model learns less.

In [10]:
from sagemaker.jumpstart.estimator import JumpStartEstimator

estimator = JumpStartEstimator(
    model_id=MODEL_ID,
    model_version=MODEL_VERSION,
    environment={"accept_eula": "true"},
    instance_type=TRAIN_INSTANCE,
    role=role,
    output_path=f"s3://{bucket}/{PREFIX}/{RUN_ID}/output",
    sagemaker_session=sess,
    tolerate_vulnerable_model=True,
)

print("Default hyperparameters for this model version:")
print(json.dumps(estimator.hyperparameters(), indent=2))

Default hyperparameters for this model version:
{
  "int8_quantization": "\"False\"",
  "enable_fsdp": "\"True\"",
  "epoch": "\"1\"",
  "learning_rate": "\"0.0001\"",
  "lora_r": "\"8\"",
  "lora_alpha": "\"32\"",
  "target_modules": "\"q_proj,v_proj\"",
  "lora_dropout": "\"0.05\"",
  "instruction_tuned": "\"False\"",
  "chat_dataset": "\"True\"",
  "add_input_output_demarcation_key": "\"True\"",
  "per_device_train_batch_size": "\"1\"",
  "per_device_eval_batch_size": "\"1\"",
  "max_train_samples": "\"-1\"",
  "max_val_samples": "\"-1\"",
  "seed": "\"10\"",
  "max_input_length": "\"-1\"",
  "validation_split_ratio": "\"0.2\"",
  "train_data_split_seed": "\"0\"",
  "preprocessing_num_workers": "\"None\"",
  "chat_template": "\"Llama3.1\""
}


In [11]:
# Names differ slightly between model versions: only the ones printed above are applied.
overrides = {
    "instruction_tuned": "True",        # our data is instruction-formatted (instruction / context / response)
    "epoch": "3",                       # 3 passes over the data: 1 pass gave too few updates to change the answers
    "learning_rate": "0.0002",          # 2e-4 is a common LoRA value
    "lora_r": "8",
    "lora_alpha": "32",
    "lora_dropout": "0.05",
    "max_input_length": "512",
    "per_device_train_batch_size": "4",
}
defaults = estimator.hyperparameters()
unknown = [k for k in overrides if k not in defaults]
print("Not recognised for this model (will be skipped):", unknown)

estimator.set_hyperparameters(**{k: v for k, v in overrides.items() if k in defaults}, chat_dataset="False")
hp = estimator.hyperparameters()
print("instruction_tuned:", hp["instruction_tuned"], "| chat_dataset:", hp["chat_dataset"])
print("epoch            :", hp["epoch"], "| learning_rate:", hp["learning_rate"])
print("batch size       :", hp["per_device_train_batch_size"], "| max_input_length:", hp["max_input_length"])
assert hp["instruction_tuned"].strip('"') == "True", "instruction_tuned must be True for this dataset."

Not recognised for this model (will be skipped): []
instruction_tuned: "True" | chat_dataset: "False"
epoch            : "3" | learning_rate: "0.0002"
batch size       : "4" | max_input_length: "512"


## 6. Run the training job

`wait=False` returns immediately so the notebook stays free. While the job runs you will deploy the base model and take a "before" baseline in section 7.

This cell will not start a second job if one is already running for this run, and it stops with a clear message if another job is using your only training slot.

In [12]:
def artifact_files_exist(job_name):
    """True if a finished job's model files are still in S3 (a lab clean-up can delete them)."""
    uri = sm.describe_training_job(TrainingJobName=job_name)["ModelArtifacts"]["S3ModelArtifacts"]
    art_bucket, art_key = uri.replace("s3://", "").split("/", 1)
    listing = s3.list_objects_v2(Bucket=art_bucket, Prefix=art_key.rstrip("/") + "/", MaxKeys=1)
    return bool(listing.get("Contents"))

current = globals().get("JOB_NAME")
current_state = None
if current:
    try:
        current_state = job_status(current)[0]
    except ClientError:
        pass

if current_state == "InProgress":
    print("A job is already running for this run, not starting another:", JOB_NAME)
elif current_state == "Completed" and artifact_files_exist(current):
    print("This run already has a completed job:", JOB_NAME)
    print("Go on to section 8. To train again, set RUN_ID = None and JOB_NAME = None, then rerun from section 3.")
else:
    if current_state == "Completed":
        print("The earlier job", current, "completed, but its model files are gone from S3. Training again.")
    if not s3.list_objects_v2(Bucket=bucket, Prefix=f"{PREFIX}/{RUN_ID}/train/", MaxKeys=1).get("Contents"):
        raise RuntimeError("The training data is no longer in S3. Run the upload cell in section 4 again, then this cell.")
    busy = jobs_with("InProgress") + jobs_with("Stopping")
    if busy:
        raise RuntimeError(f"Your training slot is in use by {busy}. Wait until it is Stopped or Completed, "
                           "or stop it with sm.stop_training_job(TrainingJobName=...), then run this cell again.")
    estimator.fit({"training": train_uri}, wait=False)
    JOB_NAME = estimator.latest_training_job.name
    print("Training job started:", JOB_NAME)


INFO:sagemaker:Creating training-job with name: llama-3-2-1b-instruct-2026-11-26-06-21-30-386


Training job started: llama-3-2-1b-instruct-2026-11-26-06-21-30-386


Watch the job in the console under **SageMaker -> Training -> Training jobs**, and open its **CloudWatch logs** to see the loss lines.

## 7. While it trains: deploy the base model (the "before" model)

To compare later, you need the **untuned** model answering the same questions. Deploying it now uses the waiting time.

> **Quota note:** quotas are counted per instance type. Hosting the base and the tuned model on the same type (for example `ml.g5.xlarge`) needs an endpoint quota of 2 for that type. If you only have quota for one, set `DEPLOY_BASE = False`. You can deploy the base model later (after you delete the tuned endpoint), or use saved baseline predictions from your instructor.
>
> **Capacity note:** if AWS has no free GPU, the endpoint ends as `Failed` with `InsufficientInstanceCapacity`. That is not your mistake. Wait 10 to 15 minutes and run this cell again; it replaces the failed endpoint.

In [13]:
from sagemaker.jumpstart.model import JumpStartModel

DEPLOY_BASE = True       # set to False if you only have quota for one endpoint

if DEPLOY_BASE:
    BASE_ENDPOINT = globals().get("BASE_ENDPOINT") or f"lab-base-{RUN_ID}"
    status, why = endpoint_status(BASE_ENDPOINT)
    if status in ("InService", "Creating", "Updating"):
        print("Base endpoint already exists:", BASE_ENDPOINT, "|", status)
    else:
        if status == "Failed":
            print("The earlier base endpoint failed:", why[:200])
            try:
                sm.delete_endpoint(EndpointName=BASE_ENDPOINT)
            except ClientError:
                pass
        if not name_is_free(BASE_ENDPOINT):          # a failed attempt can leave the name in use
            BASE_ENDPOINT = f"lab-base-{RUN_ID}-{time.strftime('%H%M%S')}"
        try:
            base_model = JumpStartModel(model_id=MODEL_ID, role=role, instance_type=ENDPOINT_INSTANCE, sagemaker_session=sess)
            base_model.deploy(
                accept_eula=True,
                initial_instance_count=1,
                endpoint_name=BASE_ENDPOINT,
                wait=False,                     # the endpoint keeps starting in the background
            )
            print("Deploying base endpoint:", BASE_ENDPOINT)
        except ClientError as error:
            print("Could not start the base endpoint:", error.response["Error"]["Code"])
            print(str(error)[:300])
            print("Continuing without it. Free an endpoint slot (Clean up) and rerun this cell later.")
            DEPLOY_BASE = False
else:
    print("Skipping the base endpoint (DEPLOY_BASE = False).")

INFO:sagemaker:Creating model with name: llama-3-2-1b-instruct-2026-11-26-06-21-41-899
INFO:sagemaker:Creating endpoint-config with name lab-base-20261126-1150
INFO:sagemaker:Creating endpoint with name lab-base-20261126-1150


Deploying base endpoint: lab-base-20261126-1150


### Wait for the training job

Expect about **15 to 25 minutes** from the moment the job really starts.

| Phase | Typical time | Notes |
|---|---|---|
| `Pending` / `Starting` | 3 to 8 minutes | SageMaker is finding a GPU. **Not billed.** |
| `Downloading` | 1 to 3 minutes | Container image, model weights and your data |
| `Training` | about 5 to 10 minutes | About 750 training rows, 3 epochs, batch size 4 |
| `Uploading` | 1 to 2 minutes | Saves the model artifact to S3 |
| `Completed` | | |

`Pending` can last much longer when AWS has no free GPU. The message column says **"Training job waiting for capacity"** in that case. It is not an error, and it is not billed. The cell stops waiting after `MAX_WAIT_MIN` minutes but the job keeps running; run the cell again to keep watching. You can also interrupt the cell at any time without stopping the job.

In [14]:
MAX_WAIT_MIN = 45          # stop waiting after this long; the job itself keeps running
CHECK_EVERY_S = 30         # seconds between status checks
start = time.time()
while True:
    d = sm.describe_training_job(TrainingJobName=JOB_NAME)
    status, secondary = d["TrainingJobStatus"], d["SecondaryStatus"]
    message = d["SecondaryStatusTransitions"][-1].get("StatusMessage", "")
    print(time.strftime("%H:%M:%S"), status, "|", secondary, "|", message)
    if status in ("Completed", "Failed", "Stopped"):
        break
    if time.time() - start > MAX_WAIT_MIN * 60:
        print(f"\nStill running after {MAX_WAIT_MIN} minutes. Run this cell again to keep waiting.")
        print("If it still says 'waiting for capacity', see the Troubleshooting section.")
        break
    time.sleep(CHECK_EVERY_S)

if status == "Completed":
    print("\nTraining finished. Billable seconds:", d.get("BillableTimeInSeconds"))
elif status in ("Failed", "Stopped"):
    print("\nJob did not complete:", d.get("FailureReason", ""))

11:51:48 InProgress | Pending | Training job waiting for capacity
11:52:18 InProgress | Pending | Training job waiting for capacity
11:52:48 InProgress | Pending | Training job waiting for capacity
11:53:19 InProgress | Pending | Training job waiting for capacity
11:53:49 InProgress | Pending | Training job waiting for capacity
11:54:20 InProgress | Pending | Training job waiting for capacity
11:54:50 InProgress | Pending | Training job waiting for capacity
11:55:21 InProgress | Pending | Training job waiting for capacity
11:55:51 InProgress | Pending | Training job waiting for capacity
11:56:23 InProgress | Pending | Training job waiting for capacity
11:56:53 InProgress | Pending | Training job waiting for capacity
11:57:24 InProgress | Pending | Training job waiting for capacity
11:57:55 InProgress | Pending | Preparing the instances for training
11:58:25 InProgress | Downloading | Downloading input data
11:58:55 InProgress | Downloading | Downloading input data
11:59:26 InProgress |

### Optional: continue a job from an earlier session

If you restarted the notebook after the job completed, set `RESUME_JOB` to its name and run the cell. This reconnects the notebook to that job. Check first that its status is `Completed` and that its files are still in S3 (section 8 verifies this).

In [15]:
RESUME_JOB = None      # for example: "llama-3-2-1b-instruct-2026-11-26-01-06-46-904"

if RESUME_JOB:
    JOB_NAME = RESUME_JOB
    estimator = JumpStartEstimator.attach(
        training_job_name=JOB_NAME, model_id=MODEL_ID, model_version=MODEL_VERSION, sagemaker_session=sess)
    print("Continuing with job:", JOB_NAME, "|", job_status()[:2])
else:
    print("Using the job from section 6:", globals().get("JOB_NAME"))

Using the job from section 6: llama-3-2-1b-instruct-2026-11-26-06-21-30-386


## 8. Save and manage model artifacts

The artifact is stored in S3 and **outlives the job and the endpoint**. This is what you version, share and deploy. Run this section only after the job status is `Completed`.

In [16]:
desc = sm.describe_training_job(TrainingJobName=JOB_NAME)
assert desc["TrainingJobStatus"] == "Completed", (
    f"The training job is {desc['TrainingJobStatus']} / {desc['SecondaryStatus']}. "
    "Wait for Completed (section 7) before running this cell.")
ARTIFACT_URI = desc["ModelArtifacts"]["S3ModelArtifacts"]
secs = desc.get("BillableTimeInSeconds", 0)

print("Artifact      :", ARTIFACT_URI)
print("Billable time :", secs, "seconds")
for m in desc.get("FinalMetricDataList", [])[:5]:
    print("Metric        :", m["MetricName"].split(":")[-1], round(m["Value"], 4))

Artifact      : s3://sagemaker-us-east-1-018913575808/finetune-lab/20261126-1150/output/llama-3-2-1b-instruct-2026-11-26-06-21-30-386/output/model
Billable time : 425 seconds
Metric        : eval-loss 0.3295
Metric        : train-loss 0.2998
Metric        : eval-ppl 1.3902


In [17]:
import pandas as pd

b, key = ARTIFACT_URI.replace("s3://", "").split("/", 1)

# For this model the artifact is a FOLDER of files (adapter weights, config), not one file,
# so list what is inside it.
objs = [o for page in s3.get_paginator("list_objects_v2").paginate(Bucket=b, Prefix=key.rstrip("/") + "/")
        for o in page.get("Contents", [])]
assert objs, (f"No files found under {ARTIFACT_URI}. The S3 folder may have been deleted; "
              "rerun sections 4 to 7 to train again.")
print(f"Artifact: {len(objs)} files, {sum(o['Size'] for o in objs) / 1e6:,.1f} MB")
for o in objs[:10]:
    print(f"  {o['Size'] / 1e6:8.2f} MB  {o['Key'].split('/')[-1]}")

# A simple run record: keep this with the artifact (a first step towards a model registry)
run_record = {
    "run_id": RUN_ID,
    "model_id": MODEL_ID,
    "training_job": JOB_NAME,
    "artifact": ARTIFACT_URI,
    "hyperparameters": estimator.hyperparameters(),
    "train_rows": sum(1 for _ in open(DATA_DIR / "train" / "train.jsonl")),
    "billable_seconds": secs,
}
pathlib.Path("runs").mkdir(exist_ok=True)
json.dump(run_record, open(f"runs/{RUN_ID}.json", "w"), indent=2)
pd.Series(run_record).to_frame("value")

Artifact: 9 files, 2,480.8 MB
      0.00 MB  __script_info__.json
      0.00 MB  config.json
      0.00 MB  generation_config.json
      0.00 MB  inference.py
   2471.65 MB  model.safetensors
      0.00 MB  special_tokens_map.json
      9.09 MB  tokenizer.json
      0.05 MB  tokenizer_config.json
      0.00 MB  version


,value
run_id,20261126-1150
model_id,meta-textgeneration-llama-3-2-1b-instruct
training_job,llama-3-2-1b-instruct-2026-11-26-06-21-30-386
artifact,s3://sagemaker-us-east-1-018913575808/finetune...
hyperparameters,"{'int8_quantization': '""False""', 'enable_fsdp'..."
train_rows,936
billable_seconds,425


**Good practice:** put the job name and data version in the S3 path, keep one run record per job (hyperparameters, data version, metrics), and never overwrite an artifact that is in use.

## 9. Deploy the fine-tuned model

`estimator.deploy()` creates the model, the endpoint configuration and the endpoint. It usually takes 5 to 10 minutes: use the time to look at the three objects in the console under **SageMaker -> Inference**.

Like the base endpoint, this cell does not create a duplicate if the endpoint already exists.

In [18]:
status, why = endpoint_status(TUNED_ENDPOINT)
if status in ("InService", "Creating", "Updating"):
    print("Tuned endpoint already exists:", TUNED_ENDPOINT, "|", status)
else:
    if status == "Failed":
        print("The earlier tuned endpoint failed:", why[:200])
        try:
            sm.delete_endpoint(EndpointName=TUNED_ENDPOINT)
        except ClientError:
            pass
    if not name_is_free(TUNED_ENDPOINT):             # a failed attempt can leave the name in use
        TUNED_ENDPOINT = f"lab-tuned-{RUN_ID}-{time.strftime('%H%M%S')}"
    deploy_args = dict(initial_instance_count=1, instance_type=ENDPOINT_INSTANCE,
                       endpoint_name=TUNED_ENDPOINT, wait=False)
    try:
        try:
            estimator.deploy(accept_eula=True, **deploy_args)
        except TypeError:                            # older SDK versions do not take accept_eula here
            estimator.deploy(**deploy_args)
        print("Deploying tuned endpoint:", TUNED_ENDPOINT)
    except ClientError as error:
        print("Could not start the tuned endpoint:", error.response["Error"]["Code"])
        print(str(error)[:300])
        print("If this is a quota error, delete an endpoint you no longer need (Clean up) and run this cell again.")

tuned_ready = wait_for_endpoint(TUNED_ENDPOINT)
if tuned_ready:
    tuned_predictor = get_predictor(TUNED_ENDPOINT)
    print("Tuned endpoint is ready:", TUNED_ENDPOINT)

INFO:sagemaker:Creating model with name: llama-3-2-1b-instruct-2026-11-26-06-35-40-759
INFO:sagemaker:Creating endpoint-config with name lab-tuned-20261126-1150
INFO:sagemaker:Creating endpoint with name lab-tuned-20261126-1150


Deploying tuned endpoint: lab-tuned-20261126-1150
12:05:50 lab-tuned-20261126-1150 | Creating
12:06:20 lab-tuned-20261126-1150 | Creating
12:06:51 lab-tuned-20261126-1150 | Creating
12:07:21 lab-tuned-20261126-1150 | Creating
12:07:52 lab-tuned-20261126-1150 | Creating
12:08:22 lab-tuned-20261126-1150 | Creating
12:08:52 lab-tuned-20261126-1150 | Creating
12:09:23 lab-tuned-20261126-1150 | Creating
12:09:53 lab-tuned-20261126-1150 | Creating
12:10:26 lab-tuned-20261126-1150 | Creating
12:10:56 lab-tuned-20261126-1150 | Creating
12:11:26 lab-tuned-20261126-1150 | Creating
12:11:57 lab-tuned-20261126-1150 | Creating
12:12:28 lab-tuned-20261126-1150 | Creating
12:12:58 lab-tuned-20261126-1150 | Creating
12:13:28 lab-tuned-20261126-1150 | Creating
12:13:59 lab-tuned-20261126-1150 | InService
Tuned endpoint is ready: lab-tuned-20261126-1150


### If the full deployment cannot finish in the session

Use a **prepared artifact** from your instructor's validated run. The pattern is to deploy the same model ID with that artifact's S3 location. Confirm the exact parameter names for your SDK version in the JumpStart documentation, since artifact handling for fine-tuned models differs between model families.

```python
# Fallback (instructor provides PREPARED_ARTIFACT_URI):
# prepared = JumpStartModel(model_id=MODEL_ID, role=role, model_data=PREPARED_ARTIFACT_URI)
# prepared.deploy(accept_eula=True, instance_type=ENDPOINT_INSTANCE, endpoint_name=TUNED_ENDPOINT)
```

If even that is not possible, skip to Notebook 3 and use the saved prediction files in its **offline mode**.

## 10. Invoke the endpoint

In [19]:
test_rows = [json.loads(l) for l in open(DATA_DIR / "test" / "test.jsonl")]
template = json.load(open(DATA_DIR / "train" / "template.json"))

def make_prompt(row):
    return template["prompt"].format(**row)

def ask(predictor, row, max_new_tokens=20):
    payload = {"inputs": make_prompt(row),
               "parameters": {"max_new_tokens": max_new_tokens, "temperature": 0.01}}
    return predictor.predict(payload, custom_attributes="accept_eula=true")

row = test_rows[0]
raw = ask(tuned_predictor, row)
print("Message  :", row["context"])
print("Expected :", row["response"])
print("Raw reply:", raw)

Message  : Where is the tracking number for the card?
Expected : card_arrival
Raw reply: {'generated_text': ' card_arrival'}


Now compare the **base** and **tuned** model on the same messages. This waits for the base endpoint from section 7; if it failed or was skipped, the cell says so and you can still continue.

In [20]:
base_ready = False
if globals().get("DEPLOY_BASE"):
    base_ready = wait_for_endpoint(BASE_ENDPOINT)
    if base_ready:
        base_predictor = get_predictor(BASE_ENDPOINT)

if base_ready:
    for r in test_rows[:5]:
        print("-" * 70)
        print("Message :", r["context"])
        print("Expected:", r["response"])
        print("Base    :", ask(base_predictor, r))
        print("Tuned   :", ask(tuned_predictor, r))
else:
    print("No base endpoint is running, so there is nothing to compare against here.")
    print("Deploy it later (section 7) or use the saved baseline predictions in Notebook 3 (offline mode).")

12:14:01 lab-base-20261126-1150 | InService
----------------------------------------------------------------------
Message : Where is the tracking number for the card?
Expected: card_arrival
Base    : {'generated_text': 'card_payment_fee_charged\n\n### Explanation:\nThe message is asking for the tracking number for a card'}
Tuned   : {'generated_text': ' card_arrival'}
----------------------------------------------------------------------
Message : Can the card was sen to me be tracked?
Expected: card_arrival
Base    : {'generated_text': 'card_arrival\n\n### Explanation:\nThe intent of this message is to ask if the card that was'}
Tuned   : {'generated_text': ' card_tracking'}
----------------------------------------------------------------------
Message : I haven't gotten my credit card in the mail.
Expected: card_arrival
Base    : {'generated_text': 'card_arrival\n\n### Explanation:\nThe intent of this message is to inform the user that their credit'}
Tuned   : {'generated_text': ' c

In [21]:
# Save the endpoint names so Notebook 3 can pick them up
json.dump({"run_id": RUN_ID, "model_id": MODEL_ID, "training_job": JOB_NAME,
           "base_endpoint": BASE_ENDPOINT if base_ready else None,
           "tuned_endpoint": TUNED_ENDPOINT, "artifact": ARTIFACT_URI, "instance": ENDPOINT_INSTANCE},
          open("endpoints.json", "w"), indent=2)
print(open("endpoints.json").read())

{
  "run_id": "20261126-1150",
  "model_id": "meta-textgeneration-llama-3-2-1b-instruct",
  "training_job": "llama-3-2-1b-instruct-2026-11-26-06-21-30-386",
  "base_endpoint": "lab-base-20261126-1150",
  "tuned_endpoint": "lab-tuned-20261126-1150",
  "artifact": "s3://sagemaker-us-east-1-018913575808/finetune-lab/20261126-1150/output/llama-3-2-1b-instruct-2026-11-26-06-21-30-386/output/model",
  "instance": "ml.g5.xlarge"
}


## 11. Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| Job stays `Pending`, message "waiting for capacity" | AWS has no free GPU of this type right now | Not an error and not billed. Wait (up to 30 to 45 minutes), or stop the job and retry later. Ask your instructor about another instance type or region. |
| `ResourceLimitExceeded` on a training job | A job is still running or `Stopping` (quota is 1) | Wait until it is `Stopped` or `Completed`, then run the cell again |
| `ResourceLimitExceeded` on an endpoint | Two endpoints already exist (quota is 2) | Delete endpoints you do not need (Clean up), then retry |
| Endpoint `Failed`, `InsufficientInstanceCapacity` | AWS has no free GPU for hosting right now | Wait 10 to 15 minutes and run the deploy cell again; it replaces the failed endpoint |
| `NoSuchBucket` during training, or section 8 finds no files | The S3 folder or bucket was deleted (lab clean-up) | Rerun sections 4 to 7 to upload and train again; ask your instructor if it repeats |
| `Cannot create already existing endpoint configuration` | A failed deploy left a config behind | Rerun the deploy cell (it picks a new name), or use the Clean up cell |
| `AccessDeniedException` | Role lacks S3 or SageMaker permission | Fix the role; check bucket name and region |
| Licence / EULA error | Gated model not accepted | Accept the licence; keep `accept_eula` set |
| Job fails immediately with a data error | Missing `template.json`, bad JSON lines, wrong channel | Re-check the `train` folder; channel is `training` |
| Out of memory | Batch size or max length too high | Lower `per_device_train_batch_size` and `max_input_length` |
| Endpoint stuck creating | Capacity or container error | Read the endpoint logs in CloudWatch; retry or change instance |

## Clean up (do not skip)

**Keep the tuned endpoint until you finish Notebook 3**, then run the next cell. Endpoints bill for every hour they exist.

With `CLEAN_UP_NOW = False` the cell only prints a plan. Read it, then set `CLEAN_UP_NOW = True` and run it again to delete.

| Resource | Deleted here? |
|---|---|
| Endpoints (`lab-base-*`, `lab-tuned-*`), their configs and models | Yes |
| A training job that is still running | Stopped |
| The S3 folder `finetune-lab/` (datasets, model artifact) | Only if you set `DELETE_S3_DATA = True` |
| Finished training job records, CloudWatch logs, the S3 bucket itself | No (kept on purpose) |

> **Shared account?** The cell finds resources by name. Check the printed plan before you delete, so you do not remove a classmate's resources.

In [22]:
CLEAN_UP_NOW = False         # False = print the plan only. True = delete.
DELETE_S3_DATA = False      # True = also delete the lab's S3 folder (datasets, model artifact)
STOP_TRAINING_JOBS = True   # stop lab training jobs that are still running

LAB_ENDPOINT_PREFIXES = ("lab-base-", "lab-tuned-")
model_stub = MODEL_ID.replace("meta-textgeneration-", "")

# ---- 1. Find what exists -------------------------------------------------
endpoint_names = {n for n in (globals().get("BASE_ENDPOINT"), globals().get("TUNED_ENDPOINT")) if n}
for page in sm.get_paginator("list_endpoints").paginate(NameContains="lab-"):
    endpoint_names |= {e["EndpointName"] for e in page["Endpoints"] if e["EndpointName"].startswith(LAB_ENDPOINT_PREFIXES)}

live_endpoints, configs, models = [], set(), set()
for ep in sorted(endpoint_names):
    try:
        d = sm.describe_endpoint(EndpointName=ep)
    except ClientError:
        continue                                   # already deleted
    live_endpoints.append(ep)
    configs.add(d["EndpointConfigName"])
    try:
        cfg_desc = sm.describe_endpoint_config(EndpointConfigName=d["EndpointConfigName"])
        models |= {v["ModelName"] for v in cfg_desc["ProductionVariants"]}
    except ClientError:
        pass                                       # config already deleted; models are also found by name below

# Leftovers from failed or repeated deploys (config or model with no endpoint)
for page in sm.get_paginator("list_endpoint_configs").paginate(NameContains="lab-"):
    configs |= {c["EndpointConfigName"] for c in page["EndpointConfigs"] if c["EndpointConfigName"].startswith(LAB_ENDPOINT_PREFIXES)}
for page in sm.get_paginator("list_models").paginate(NameContains=model_stub):
    models |= {m["ModelName"] for m in page["Models"]}

running_jobs = []
if STOP_TRAINING_JOBS:
    for page in sm.get_paginator("list_training_jobs").paginate(StatusEquals="InProgress", NameContains=model_stub):
        running_jobs += [j["TrainingJobName"] for j in page["TrainingJobSummaries"]]

s3_objects = []
if DELETE_S3_DATA:
    try:
        for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=f"{PREFIX}/"):
            s3_objects += [{"Key": o["Key"], "Size": o["Size"]} for o in page.get("Contents", [])]
    except ClientError as error:
        print("Could not list S3 data:", repr(error)[:120])

# ---- 2. Show the plan ----------------------------------------------------
print("PLAN" + ("  (nothing is deleted yet)" if not CLEAN_UP_NOW else ""))
print("Endpoints         :", live_endpoints or "none")
print("Endpoint configs  :", sorted(configs) or "none")
print("Models            :", sorted(models) or "none")
print("Running jobs      :", running_jobs or "none")
if DELETE_S3_DATA:
    print(f"S3 data           : {len(s3_objects)} objects, {sum(o['Size'] for o in s3_objects) / 1e6:,.1f} MB in s3://{bucket}/{PREFIX}/")
else:
    print("S3 data           : kept (set DELETE_S3_DATA = True to delete)")

# ---- 3. Delete -----------------------------------------------------------
def attempt(label, fn, **kwargs):
    try:
        fn(**kwargs)
        print("Deleted/stopped:", label)
    except ClientError as error:
        print("Could not process", label, "->", repr(error)[:140])

if CLEAN_UP_NOW:
    print()
    for job in running_jobs:
        attempt(f"training job {job}", sm.stop_training_job, TrainingJobName=job)
    for ep in live_endpoints:
        attempt(f"endpoint {ep}", sm.delete_endpoint, EndpointName=ep)
    for c in sorted(configs):
        attempt(f"endpoint config {c}", sm.delete_endpoint_config, EndpointConfigName=c)
    for m in sorted(models):
        attempt(f"model {m}", sm.delete_model, ModelName=m)
    if DELETE_S3_DATA and s3_objects:
        for i in range(0, len(s3_objects), 1000):
            s3.delete_objects(Bucket=bucket, Delete={"Objects": [{"Key": o["Key"]} for o in s3_objects[i:i + 1000]]})
        print(f"Deleted {len(s3_objects)} S3 objects under s3://{bucket}/{PREFIX}/")
    print("\nDone. Check the console: SageMaker -> Inference -> Endpoints should be empty.")
    print("An endpoint that is still starting cannot be deleted yet: run this cell again in a few minutes.")
else:
    print("\nSet CLEAN_UP_NOW = True and run this cell again to delete.")

PLAN  (nothing is deleted yet)
Endpoints         : ['lab-base-20261126-1150', 'lab-tuned-20261126-1150']
Endpoint configs  : ['lab-base-20261126-1150', 'lab-tuned-20261126-1150']
Models            : ['llama-3-2-1b-instruct-2026-11-26-06-21-41-899', 'llama-3-2-1b-instruct-2026-11-26-06-35-40-759']
Running jobs      : none
S3 data           : kept (set DELETE_S3_DATA = True to delete)

Set CLEAN_UP_NOW = True and run this cell again to delete.


Then check the console: **Inference -> Endpoints** should be empty, and stop any Studio apps you no longer need.

## Try it

1. Change `epoch` to 3 and `lora_r` to 16 in `overrides`, and run a second job (set `RUN_ID = None` and `JOB_NAME = None` first). How do billable time and the artifact size change?
2. Find the training job's CloudWatch log stream and locate the loss values.
3. Calculate the cost of your job: billable seconds / 3600 x the instance's hourly price.
4. Why do we save a run record next to the artifact?

In [23]:
# Your answers here

## Next

Open **03_Evaluation_And_Comparison.ipynb** to score the base and tuned models on the same test set.